# Verify the Completed Audit Capture

Run this notebook only in the Audit environment, after the experiment kernel has been shut down and Sciunit has finalized execution `e1`. It inspects the captured inputs and outputs without modifying the package.

Always select the regular **Python 3 (ipykernel)** for this notebook. Do not select a Sciunit kernel.

## 1. Initialize the Audit Check

This cell prepares the Audit verification. It defines read-only helpers and the expected results for the four-file demonstration. No settings need to be changed.

In [ ]:
from datetime import datetime
from pathlib import Path
from IPython.display import Image, Markdown, display
import os
import re
import subprocess

VERIFICATION_POINT = "Audit - after capture"
EXPECTED_INPUT_FILES = 4
EXPECTED_PLOT_FILES = 4
MAX_PLOTS_TO_DISPLAY = 4
SCIUNIT_PROJECT_OVERRIDE = ""

SCIUNIT_HOME = Path.home() / "sciunit"
LIVE_WORKSPACE = Path("/workspace")

def human_size(size_bytes: int) -> str:
    value = float(size_bytes)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if value < 1024 or unit == "TiB":
            return f"{value:.1f} {unit}"
        value /= 1024

def execution_number(path: Path) -> int:
    match = re.fullmatch(r"e(\d+)\.json", path.name)
    return int(match.group(1)) if match else 10**9

print(f"Verification checkpoint: {VERIFICATION_POINT}")
print(f"Session mode: {os.environ.get('SESSION_MODE', 'not provided')}")
print(f"Inspection time: {datetime.now().isoformat(timespec='seconds')}")

## 2. Locate the Active Sciunit Project

Sciunit records the currently opened project in `~/sciunit/.activated`. This cell reads that location, confirms that the CDE package exists, lists captured execution records such as `e1`, and locates the captured `/workspace` filesystem.

If `e1` is missing, return to the experiment notebook and shut down its Audit kernel before continuing.

In [ ]:
checkout = subprocess.run(
    ["sciunit", "checkout", "e1"],
    check=False,
    capture_output=True,
    text=True,
)
if checkout.stdout.strip():
    print(checkout.stdout.strip())
if checkout.stderr.strip():
    print(checkout.stderr.strip())
if checkout.returncode != 0:
    raise RuntimeError(
        "Sciunit could not check out execution e1. Confirm that the Audit "
        "experiment kernel was shut down and e1 was finalized."
    )

def active_sciunit_project() -> Path:
    if SCIUNIT_PROJECT_OVERRIDE:
        return Path(SCIUNIT_PROJECT_OVERRIDE).expanduser().resolve()

    activated_file = SCIUNIT_HOME / ".activated"
    if not activated_file.is_file():
        raise RuntimeError(
            "No active Audit project was found. Finish the Audit run and "
            "shut down its experiment kernel first."
        )

    active_paths = [line.strip() for line in activated_file.read_text().splitlines() if line.strip()]
    if not active_paths:
        raise RuntimeError(f"The Sciunit activation file is empty: {activated_file}")
    return Path(active_paths[0]).expanduser().resolve()

PROJECT_PATH = active_sciunit_project()
CDE_PACKAGE = PROJECT_PATH / "cde-package"
CDE_ROOT = CDE_PACKAGE / "cde-root"
CAPTURED_WORKSPACE = CDE_ROOT / "workspace"

if not CDE_PACKAGE.is_dir():
    raise RuntimeError(f"The active project does not contain a CDE package: {CDE_PACKAGE}")
if not CAPTURED_WORKSPACE.is_dir():
    raise RuntimeError(f"The captured /workspace directory was not found: {CAPTURED_WORKSPACE}")

EXECUTION_FILES = sorted(PROJECT_PATH.glob("e*.json"), key=execution_number)

print(f"Active project: {PROJECT_PATH}")
print(f"CDE package: {CDE_PACKAGE}")
print(f"Captured workspace: {CAPTURED_WORKSPACE}")
print("Execution records:", ", ".join(path.stem for path in EXECUTION_FILES) or "none")

## 3. Inspect Captured IMERG Inputs

The experiment reads NetCDF files through `/workspace/data/discover/<YYYYMM>/`. During Audit, Sciunit captures the files that were actually opened. Inside the package, that virtual path is stored under `cde-package/cde-root/workspace/data/discover/`.

This cell lists every captured `.nc4` input with its relative path and size. For the default demonstration, expect four files under `202001`.

In [ ]:
CAPTURED_INPUT_ROOT = CAPTURED_WORKSPACE / "data" / "discover"
CAPTURED_INPUT_FILES = sorted(
    path for path in CAPTURED_INPUT_ROOT.rglob("*.nc4") if path.is_file()
) if CAPTURED_INPUT_ROOT.is_dir() else []

print(f"Captured input directory: {CAPTURED_INPUT_ROOT}")
print(f"Captured NetCDF files: {len(CAPTURED_INPUT_FILES)}")
for path in CAPTURED_INPUT_FILES:
    print(f"  {path.relative_to(CAPTURED_INPUT_ROOT)}  ({human_size(path.stat().st_size)})")

if len(CAPTURED_INPUT_FILES) < EXPECTED_INPUT_FILES:
    print(f"WARNING: Expected at least {EXPECTED_INPUT_FILES} captured inputs.")
else:
    print("PASS: The expected IMERG input files are present in the CDE package.")

## 4. Inspect Generated Output Files

This cell checks the output files created in the live Jupyter workspace. It groups files by workflow stage and reports NumPy array shapes without loading complete arrays into memory.

The expected four-file outputs are precipitation masks, per-timestamp connected-component arrays, a combined component file, per-timestamp tracked arrays, and four PNG maps.

In [ ]:
import numpy as np

OUTPUT_CATEGORIES = {
    "Precipitation masks": "dyamond_raw_pr_*.npy",
    "Per-timestamp CCL arrays": "get_ccl_*.npy",
    "Combined CCL files": "dyamond_ccl_*.pkl",
    "Tracked CCL arrays": "ccl_map_*.npy",
    "Tracked PNG maps": "tracked_ccl_map_*.png",
}

OUTPUT_LOCATIONS = {
    "Jupyter workspace": LIVE_WORKSPACE / "output",
}
OUTPUT_INVENTORY = {}

for location_name, output_root in OUTPUT_LOCATIONS.items():
    print(f"\n{location_name}: {output_root}")
    location_inventory = {}
    for category, pattern in OUTPUT_CATEGORIES.items():
        files = sorted(path for path in output_root.glob(pattern) if path.is_file()) if output_root.is_dir() else []
        location_inventory[category] = files
        print(f"  {category}: {len(files)}")
        for path in files:
            details = human_size(path.stat().st_size)
            if path.suffix == ".npy":
                try:
                    array = np.load(path, mmap_mode="r")
                    details += f", shape={array.shape}, dtype={array.dtype}"
                    del array
                except Exception as exc:
                    details += f", array metadata unavailable: {exc}"
            print(f"    {path.name}  ({details})")
    OUTPUT_INVENTORY[location_name] = location_inventory

## 5. Display the Generated Maps

This cell displays up to four tracked PNG maps directly from the live Jupyter workspace.

Confirm that there is one readable map for each processed timestamp and that the timestamp title advances in chronological order.

In [ ]:
live_plots = OUTPUT_INVENTORY["Jupyter workspace"]["Tracked PNG maps"]
PLOTS_TO_SHOW = live_plots
PLOT_SOURCE = "Jupyter workspace"

if not PLOTS_TO_SHOW:
    print("No tracked PNG maps were found. Complete the experiment before checking plots.")
else:
    display(Markdown(f"### Plot source: {PLOT_SOURCE}"))
    for plot_path in PLOTS_TO_SHOW[:MAX_PLOTS_TO_DISPLAY]:
        display(Markdown(f"**{plot_path.name}**"))
        display(Image(filename=str(plot_path), width=900))

## 6. Verification Summary

The final cell checks execution `e1`, captured IMERG inputs, and the expected generated artifacts in the Jupyter workspace. A warning identifies a missing item but does not change any files.

Passing checks confirm that Audit created `e1`, captured the expected four-file inputs, and generated the expected workspace outputs.

In [ ]:
def workspace_output_count(category: str) -> int:
    return len(OUTPUT_INVENTORY["Jupyter workspace"][category])

checks = [
    ("Execution e1 exists", any(path.name == "e1.json" for path in EXECUTION_FILES)),
    (f"At least {EXPECTED_INPUT_FILES} captured IMERG inputs", len(CAPTURED_INPUT_FILES) >= EXPECTED_INPUT_FILES),
    ("Workspace precipitation mask exists", workspace_output_count("Precipitation masks") >= 1),
    (f"At least {EXPECTED_INPUT_FILES} workspace CCL arrays", workspace_output_count("Per-timestamp CCL arrays") >= EXPECTED_INPUT_FILES),
    ("Workspace combined CCL file exists", workspace_output_count("Combined CCL files") >= 1),
    (f"At least {EXPECTED_INPUT_FILES} workspace tracked arrays", workspace_output_count("Tracked CCL arrays") >= EXPECTED_INPUT_FILES),
    (f"At least {EXPECTED_PLOT_FILES} workspace tracked maps", workspace_output_count("Tracked PNG maps") >= EXPECTED_PLOT_FILES),
]

display(Markdown(f"## {VERIFICATION_POINT}"))
for description, passed in checks:
    print(f"{'PASS' if passed else 'WARNING'}: {description}")

if all(passed for _, passed in checks):
    print("\nVerification complete: all expected four-file experiment artifacts are available.")
else:
    print("\nVerification found missing artifacts. Review the warnings and the inventory above.")

## 7. Create the Sciunit Share Link

Run this final cell after the verification checks pass. It runs `sciunit copy` for the finalized `e1` execution and prints the CloudFront link. Copy the printed link for the Repeat environment. This completes the Audit workflow; then use **Stop Environment** in the portal.

In [ ]:
copy_result = subprocess.run(
    ["sciunit", "copy"],
    check=False,
    capture_output=True,
    text=True,
)
if copy_result.stdout.strip():
    print(copy_result.stdout.strip())
if copy_result.stderr.strip():
    print(copy_result.stderr.strip())
if copy_result.returncode != 0:
    raise RuntimeError("sciunit copy failed. Review the command output above.")
